# Scraping YouTube Comments / Data Acquisition
**Nama**: Asep Haryana Saputra
**Project**: Sentiment Analysis (YouTube Comments)

This notebook contains two methods for acquiring data:
1. **Scraping**: Direct scraping from YouTube (may be rate-limited).
2. **Benchmark Dataset (Fallback)**: Downloading the IndonLU benchmark dataset (high quality, 3 classes).

In [ ]:
import pandas as pd
from youtube_comment_downloader import YoutubeCommentDownloader
import itertools
import os
import time

# Configuration
VIDEO_URL = 'https://www.youtube.com/watch?v=k9a5LpSfp7c'
TARGET_COUNT = 15000 
OUTPUT_FILE = 'dataset_youtube.csv'

## Method 1: YouTube Scraping (Incremental Save)

In [ ]:
def scrape_youtube_comments(url, limit=100, output_file='dataset_youtube.csv'):
    downloader = YoutubeCommentDownloader()
    try:
        comments = downloader.get_comments_from_url(url, sort_by=1) # sort_by=1 (recents), 0 (popular)
    except Exception as e:
        print(f"Error initializing downloader: {e}")
        return

    print(f"Starting scraping for {url}...")
    
    count = 0
    batch_data = []
    
    # Initialize file with header
    if not os.path.exists(output_file):
        pd.DataFrame(columns=['cid', 'text', 'time', 'author', 'votes', 'photo', 'heart']).to_csv(output_file, index=False)

    for comment in itertools.islice(comments, limit):
        batch_data.append({
            'cid': comment['cid'],
            'text': comment['text'],
            'time': comment['time'],
            'author': comment['author'],
            'votes': comment.get('votes', 0),
            'photo': comment.get('photo', ''),
            'heart': comment.get('heart', False)
        })
        count += 1
        
        if count % 100 == 0:
            # Append to CSV
            df_batch = pd.DataFrame(batch_data)
            df_batch.to_csv(output_file, mode='a', header=False, index=False)
            batch_data = [] # Clear batch
            print(f"Scraped {count}/{limit} comments...", end='\r')

    # Save remaining
    if batch_data:
        df_batch = pd.DataFrame(batch_data)
        df_batch.to_csv(output_file, mode='a', header=False, index=False)
        
    print(f"\nTotal scraped: {count}")

# Uncomment to run scraping:
# scrape_youtube_comments(VIDEO_URL, limit=TARGET_COUNT, output_file=OUTPUT_FILE)

## Method 2: IndonLU Benchmark Dataset (Recommended)
If scraping fails or is too slow, we use the IndonLU SMSA dataset (11k samples).

In [ ]:
def download_and_convert_benchmark():
    url = "https://raw.githubusercontent.com/indobenchmark/indonlu/master/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv"
    print("Downloading dataset...")
    
    # Load TSV directly
    try:
        df = pd.read_csv(url, sep='\t', names=['text', 'label_str'])
        
        # Map labels to integers
        label_map = {'negative': 0, 'positive': 1, 'neutral': 2}
        df['label'] = df['label_str'].map(label_map)
        
        # Save as if it were the scraped dataset
        output = 'dataset_youtube.csv'
        df[['text', 'label']].to_csv(output, index=False)
        print(f"Successfully downloaded {len(df)} rows and saved to {output}")
        print(df['label'].value_counts())
        
    except Exception as e:
        print(f"Error downloading: {e}")

# Run this if scraping is skipped
if not os.path.exists(OUTPUT_FILE):
    download_and_convert_benchmark()
else:
    print(f"{OUTPUT_FILE} already exists.")